# Expected Shortfall (CVaR)

This notebook demonstrates **Expected Shortfall (ES)**, also called **Conditional Value-at-Risk (CVaR)**.

While VaR gives the minimum expected loss at a given confidence level (e.g. 95%), ES gives the *average loss in the worst 5% of cases*. 

We will:
1. Load asset returns
2. Compute Historical VaR
3. Compute Historical ES
4. Compare the two measures

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from src.var_methods import historical_var
from src.es_methods import historical_es

## Load Data
We use the same SPY ETF example as in the previous notebook.

In [ ]:
ticker = "SPY"
data = yf.download(ticker, start="2020-01-01", end="2025-01-01")["Close"]
log_returns = np.log(data / data.shift(1)).dropna()

log_returns.head()

## Historical VaR vs. ES

- We assume a portfolio value of **$1,000,000**
- Significance level **α = 0.05** (95% confidence)

In [ ]:
portfolio_value = 1_000_000
alpha = 0.05

hist_var = historical_var(log_returns, alpha=alpha, portfolio_value=portfolio_value)
hist_es = historical_es(log_returns.to_frame(), weights=np.array([1.0]), alpha=alpha, portfolio_value=portfolio_value)

print(f"Historical 95% VaR: ${hist_var:,.2f}")
print(f"Historical 95% ES:  ${hist_es:,.2f}")

## Visualization
We highlight the **VaR cutoff** (red dashed line) and the **tail region** used for ES (shaded area).

In [ ]:
cutoff = np.percentile(log_returns.values, alpha*100)
tail_losses = log_returns[log_returns <= cutoff]

plt.figure(figsize=(10,6))
plt.hist(log_returns, bins=50, alpha=0.6, label="Returns")
plt.axvline(cutoff, color="red", linestyle="--", label="95% VaR cutoff")
plt.hist(tail_losses, bins=30, alpha=0.6, color="orange", label="Tail losses (ES)")
plt.title("Distribution of SPY Daily Returns with 95% VaR and ES Tail")
plt.xlabel("Daily Log Return")
plt.ylabel("Frequency")
plt.legend()
plt.show()

## Summary

- Historical VaR gives the **threshold loss** exceeded on 5% of days.
- Historical ES (CVaR) gives the **average loss** on those worst 5% of days.
- ES is considered a more robust and *coherent* risk measure, since it accounts for tail severity, not just frequency.